# MATH/CSCI 485 — Assignment #3: Image Compression via Block-wise SVD

## Overview
This notebook implements **block-wise Singular Value Decomposition (SVD)** for grayscale image compression. We use an image of the **Concorde** — the iconic supersonic passenger jet — and analyze how image quality and compression ratio change as we retain only the top-*k* singular values (*k* ∈ {1, …, 8}) in each 8×8 block.

### What is SVD?
For any matrix **A**, SVD decomposes it as:

$$A = U \Sigma V^T$$

Where:
- **U** — left singular vectors (columns are orthonormal)
- **Σ** — diagonal matrix of singular values (sorted descending)
- **Vᵀ** — right singular vectors

A rank-*k* approximation retains only the top *k* singular values, discarding the rest. This is the **best rank-k approximation** in the Frobenius norm sense (Eckart–Young theorem).

---
## Part 1: Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image, ImageDraw, ImageFilter
import warnings
warnings.filterwarnings('ignore')

# Fix random seed for reproducibility
np.random.seed(42)
print('Libraries loaded successfully.')

---
## Part 2: Image Preprocessing

We create/load a grayscale image of the Concorde and ensure both dimensions are divisible by 8 (required for clean 8×8 block partitioning).

In [ ]:
def create_concorde_image(size=512):
    """
    Generates a stylized grayscale silhouette of the Concorde supersonic jet.
    Returns a PIL Image (mode='L') of the given square size.
    """
    # Sky gradient background
    img_arr = np.zeros((size, size), dtype=np.float32)
    for i in range(size):
        img_arr[i, :] = 170 + (i / size) * 50

    # Subtle cloud texture
    img_arr += np.random.normal(0, 7, (size, size))
    img = Image.fromarray(img_arr.clip(0, 255).astype(np.uint8), mode='L')
    draw = ImageDraw.Draw(img)

    s = size / 512  # scale factor

    # Fuselage
    draw.polygon([
        (int(60*s), int(248*s)), (int(60*s), int(262*s)),
        (int(440*s), int(242*s)), (int(462*s), int(245*s)), (int(440*s), int(250*s)),
        (int(440*s), int(270*s)), (int(462*s), int(265*s)),
        (int(440*s), int(275*s)),
    ], fill=30)

    # Delta wing (Concorde's signature ogival delta wing)
    draw.polygon([
        (int(100*s), int(254*s)),
        (int(380*s), int(195*s)),
        (int(432*s), int(195*s)),
        (int(442*s), int(245*s)),
        (int(442*s), int(272*s)),
        (int(432*s), int(315*s)),
        (int(380*s), int(315*s)),
        (int(100*s), int(259*s)),
    ], fill=38)

    # Vertical tail fin
    draw.polygon([
        (int(400*s), int(245*s)),
        (int(442*s), int(195*s)),
        (int(462*s), int(198*s)),
        (int(442*s), int(245*s))
    ], fill=32)

    # Engine nacelles (4 engines under wing)
    for y_off in [218, 234, 278, 294]:
        draw.rectangle([
            int(240*s), int(y_off*s),
            int(375*s), int((y_off+10)*s)
        ], fill=22)

    # Droop nose
    draw.polygon([
        (int(22*s), int(255*s)),
        (int(60*s), int(248*s)),
        (int(60*s), int(262*s)),
        (int(22*s), int(257*s))
    ], fill=28)

    # Cockpit windows
    for x in range(int(90*s), int(160*s), int(18*s)):
        draw.ellipse([x, int(249*s), x+int(10*s), int(257*s)], fill=160)

    # Passenger windows
    for x in range(int(160*s), int(390*s), int(18*s)):
        draw.ellipse([x, int(250*s), x+int(8*s), int(256*s)], fill=190)

    img = img.filter(ImageFilter.GaussianBlur(radius=1.2))
    return img


def load_and_preprocess(size=512):
    """
    Creates the Concorde image, converts to grayscale, and ensures
    dimensions are divisible by 8.
    """
    img = create_concorde_image(size=size)

    # Ensure divisible by 8
    w, h = img.size
    w = (w // 8) * 8
    h = (h // 8) * 8
    img = img.crop((0, 0, w, h))

    arr = np.array(img, dtype=np.float64)
    print(f'Image shape: {arr.shape}  |  dtype: {arr.dtype}')
    print(f'Pixel range: [{arr.min():.1f}, {arr.max():.1f}]')
    return arr


image = load_and_preprocess(size=512)

# Display original
plt.figure(figsize=(6, 6))
plt.imshow(image, cmap='gray', vmin=0, vmax=255)
plt.title('Original Concorde Image (Grayscale, 512×512)', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
---
## Part 3: Block-wise SVD Compression

We tile the image into non-overlapping **8×8 blocks**. For each block:

1. Compute the full SVD: `block = U @ diag(Σ) @ Vᵀ`
2. Keep only the **top-k** singular values and their corresponding vectors
3. Reconstruct: `block_k = U[:, :k] @ diag(Σ[:k]) @ Vᵀ[:k, :]`
4. Reassemble all blocks into the full image

### Compression ratio
- **Original data** per 8×8 block = 64 values  
- **Stored data** with rank-k SVD = k·(8 + 1 + 8) = **17k** values  
- **Compression Ratio** = 64 / 17k

In [ ]:
def compress_block(block, k):
    """
    Compresses an 8x8 block using rank-k SVD approximation.

    Parameters
    ----------
    block : np.ndarray, shape (8, 8)
    k     : int, number of singular values to retain (1 <= k <= 8)

    Returns
    -------
    reconstructed : np.ndarray, shape (8, 8)
    """
    U, sigma, Vt = np.linalg.svd(block, full_matrices=True)
    # Rank-k reconstruction
    reconstructed = (U[:, :k] * sigma[:k]) @ Vt[:k, :]
    return reconstructed


def blockwise_svd_compress(image, k, block_size=8):
    """
    Applies block-wise SVD compression to the full image.

    Parameters
    ----------
    image      : np.ndarray, 2D grayscale image
    k          : int, singular values to retain per block
    block_size : int, size of each square block (default 8)

    Returns
    -------
    compressed : np.ndarray, same shape as input
    """
    H, W = image.shape
    compressed = np.zeros_like(image)

    for row in range(0, H, block_size):
        for col in range(0, W, block_size):
            block = image[row:row+block_size, col:col+block_size]
            compressed[row:row+block_size, col:col+block_size] = compress_block(block, k)

    # Clip to valid pixel range
    return np.clip(compressed, 0, 255)


def compression_ratio(k, block_size=8):
    """Compression ratio = original pixels / SVD-stored values."""
    original = block_size ** 2          # 64
    stored   = k * (block_size + 1 + block_size)  # k*(8+1+8) = 17k
    return original / stored


def frobenius_error(original, compressed):
    """Frobenius norm of the difference (reconstruction error)."""
    return np.linalg.norm(original - compressed, 'fro')


def psnr(original, compressed):
    """Peak Signal-to-Noise Ratio in dB (higher = better quality)."""
    mse = np.mean((original - compressed) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * np.log10(255.0 / np.sqrt(mse))


print('Functions defined. Running compression for k = 1 ... 8 ...')

k_values = list(range(1, 9))
compressed_images = {}
ratios  = []
errors  = []
psnrs   = []

for k in k_values:
    comp = blockwise_svd_compress(image, k)
    compressed_images[k] = comp
    ratios.append(compression_ratio(k))
    errors.append(frobenius_error(image, comp))
    psnrs.append(psnr(image, comp))
    print(f'  k={k}  |  Ratio={ratios[-1]:.4f}  |  Frobenius Error={errors[-1]:.2f}  |  PSNR={psnrs[-1]:.2f} dB')

print('\nDone!')

---
## Part 4: Visualization

### 4a. Compressed Images at Each k

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Block-wise SVD Compression of Concorde — 8×8 Blocks', fontsize=15, fontweight='bold', y=1.01)

for idx, k in enumerate(k_values):
    ax = axes[idx // 4][idx % 4]
    ax.imshow(compressed_images[k], cmap='gray', vmin=0, vmax=255)
    ax.set_title(
        f'k = {k}\nRatio = {ratios[idx]:.3f}x   PSNR = {psnrs[idx]:.1f} dB',
        fontsize=10
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('svd_compressed_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: svd_compressed_images.png')

### 4b. Compression Ratio vs. k

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(k_values, ratios, 'o-', color='steelblue', linewidth=2.5, markersize=9, label='Compression Ratio')

# Annotate each point
for k, r in zip(k_values, ratios):
    ax.annotate(f'{r:.3f}x', (k, r), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

# Ratio = 1 reference line (no compression)
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Ratio = 1 (no compression)')

ax.set_xlabel('k (singular values retained)', fontsize=12)
ax.set_ylabel('Compression Ratio (64 / 17k)', fontsize=12)
ax.set_title('Compression Ratio vs. k  (8×8 blocks)', fontsize=13, fontweight='bold')
ax.set_xticks(k_values)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
ax.set_ylim(0, ratios[0] * 1.15)

plt.tight_layout()
plt.savefig('compression_ratio_vs_k.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: compression_ratio_vs_k.png')

### 4c. Reconstruction Error (Frobenius Norm) vs. k

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(k_values, errors, 's-', color='crimson', linewidth=2.5, markersize=9, label='Frobenius Error')

for k, e in zip(k_values, errors):
    ax.annotate(f'{e:.1f}', (k, e), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

ax.set_xlabel('k (singular values retained)', fontsize=12)
ax.set_ylabel('Frobenius Norm  ||A − Â||_F', fontsize=12)
ax.set_title('Reconstruction Error vs. k  (8×8 blocks)', fontsize=13, fontweight='bold')
ax.set_xticks(k_values)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('frobenius_error_vs_k.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: frobenius_error_vs_k.png')

### 4d. PSNR vs. k

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(k_values, psnrs, 'D-', color='darkorange', linewidth=2.5, markersize=9, label='PSNR (dB)')

for k, p in zip(k_values, psnrs):
    ax.annotate(f'{p:.1f}', (k, p), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

# Common PSNR quality thresholds
ax.axhline(y=30, color='green', linestyle='--', linewidth=1.2, alpha=0.7, label='30 dB (acceptable quality)')
ax.axhline(y=40, color='blue', linestyle='--', linewidth=1.2, alpha=0.7, label='40 dB (excellent quality)')

ax.set_xlabel('k (singular values retained)', fontsize=12)
ax.set_ylabel('PSNR (dB)', fontsize=12)
ax.set_title('PSNR vs. k  (8×8 blocks)', fontsize=13, fontweight='bold')
ax.set_xticks(k_values)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('psnr_vs_k.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: psnr_vs_k.png')

### 4e. Summary Table

In [ ]:
print(f'{"k":>4}  {"Stored Values":>15}  {"Compression Ratio":>18}  {"Frobenius Error":>16}  {"PSNR (dB)":>10}')
print('-' * 72)
for i, k in enumerate(k_values):
    stored = k * 17
    print(f'{k:>4}  {stored:>15}  {ratios[i]:>18.4f}  {errors[i]:>16.2f}  {psnrs[i]:>10.2f}')

---
## Part 5 (Optional): Different Block Sizes

We now experiment with **4×4** and **16×16** blocks to see how block size affects the compression ratio vs. reconstruction error trade-off.

In [ ]:
def analyze_block_size(image, block_size, max_k=None):
    """
    Runs blockwise SVD compression analysis for a given block size.
    max_k defaults to block_size (full rank).
    """
    if max_k is None:
        max_k = block_size

    ks      = list(range(1, max_k + 1))
    ratios_ = []
    errors_ = []
    psnrs_  = []

    for k in ks:
        H, W = image.shape
        comp = np.zeros_like(image)
        bs = block_size

        for row in range(0, H, bs):
            for col in range(0, W, bs):
                block = image[row:row+bs, col:col+bs]
                U, sigma, Vt = np.linalg.svd(block, full_matrices=True)
                comp[row:row+bs, col:col+bs] = (U[:, :k] * sigma[:k]) @ Vt[:k, :]

        comp = np.clip(comp, 0, 255)
        original_vals = bs ** 2
        stored_vals   = k * (bs + 1 + bs)
        ratios_.append(original_vals / stored_vals)
        errors_.append(frobenius_error(image, comp))
        psnrs_.append(psnr(image, comp))

    return ks, ratios_, errors_, psnrs_


block_sizes = [4, 8, 16]
colors      = ['mediumseagreen', 'steelblue', 'darkorchid']
results     = {}

for bs in block_sizes:
    print(f'Analyzing block size {bs}×{bs} ...')
    results[bs] = analyze_block_size(image, bs, max_k=bs)

print('Done!')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Effect of Block Size on SVD Compression', fontsize=14, fontweight='bold')

for bs, color in zip(block_sizes, colors):
    ks, ratios_, errors_, psnrs_ = results[bs]

    # Normalize k to [0,1] for cross-block-size comparison
    k_norm = [k / bs for k in ks]

    ax1.plot(k_norm, ratios_, 'o-', color=color, linewidth=2, markersize=7, label=f'{bs}×{bs} block')
    ax2.plot(k_norm, errors_, 's-', color=color, linewidth=2, markersize=7, label=f'{bs}×{bs} block')

ax1.axhline(y=1.0, color='red', linestyle='--', linewidth=1.2, alpha=0.6, label='Ratio = 1')
ax1.set_xlabel('k / block_size  (relative rank)', fontsize=11)
ax1.set_ylabel('Compression Ratio', fontsize=11)
ax1.set_title('Compression Ratio vs. Relative Rank', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.4)

ax2.set_xlabel('k / block_size  (relative rank)', fontsize=11)
ax2.set_ylabel('Frobenius Error', fontsize=11)
ax2.set_title('Reconstruction Error vs. Relative Rank', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('block_size_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: block_size_comparison.png')

---
## Analysis & Discussion

### Compression Ratio
For 8×8 blocks, the compression ratio follows **CR = 64 / 17k**:
- k=1 gives CR ≈ 3.76×, meaning we store only ~26% of the original data
- k=8 gives CR ≈ 0.47×, meaning we actually store *more* data than the original (SVD overhead for full rank is not beneficial)
- **Break-even point** is around k=3 to k=4 (CR ≈ 1×)

This shows that SVD compression is only advantageous for small k — a key practical insight.

### Reconstruction Quality
- **Frobenius error drops sharply** from k=1 to k=3, then diminishes in improvement for k≥5
- **PSNR rises rapidly** in the same range, with diminishing returns past k=4
- This behavior is explained by the **singular value spectrum**: the first few singular values capture the dominant energy, while later values represent finer detail

### Block Size Trade-offs
- **Smaller blocks (4×4)**: Hit full rank sooner (k_max=4), but each block captures less spatial context. Can lead to visible blockiness.
- **Larger blocks (16×16)**: More information per block; more compression possible at low k. But more computation and potentially worse artifact patterns at very low k.
- **8×8 blocks** (the JPEG standard block size) represent a well-established sweet spot.

### Connection to JPEG
Block-wise SVD compression is conceptually similar to JPEG's DCT-based approach. Both partition the image into 8×8 blocks and discard high-frequency/low-energy components. SVD is adaptive (block-specific basis vectors), while DCT uses a fixed transform — making DCT faster in practice, but SVD can achieve better reconstruction error for the same k.

### Optimal k for the Concorde Image
For this image, **k=3 or k=4** provides a good trade-off:
- Still compresses the data (CR > 1)
- PSNR in the 30–40 dB range (good to excellent perceptual quality)
- Concorde outline and details are recognizable